In [199]:
import os
import pandas as pd
import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta
from scipy.interpolate import CubicSpline
from scipy.optimize import brentq

# MODULE 0

In [200]:
def cpn_schedule(maturity, settle):
    """Function to get the coupon schedule"""
    if settle >= maturity:
        raise ValueError("settle must be before maturity")
    k = 0
    while maturity - relativedelta(months=6 * (k + 1)) > settle:
        k += 1
    n = k + 1                                         # coupons strictly after settle
    next_cpn = maturity - relativedelta(months=6 * k)
    prev_cpn = maturity - relativedelta(months=6 * (k + 1))
    return prev_cpn, next_cpn, n

In [201]:
def gilt_price(coupon, ytm, ex_div_bdays=7,\
    settle=date(2026, 10, 10), maturity=date(2055, 10, 10)):
    """Function to get clean, dirty, accrued, dv01 (per 100 nominal)"""
    prev_cpn, next_cpn, n = cpn_schedule(maturity, settle)
    t = (next_cpn-settle).days # days from settlement to next coupon
    s = (next_cpn-prev_cpn).days # days in current coupon period

    ex_div_date = (pd.Timestamp(next_cpn)-pd.offsets.BDay(ex_div_bdays)).date()
    is_ex = settle>=ex_div_date
    if is_ex:
        ai = -(coupon/2)*(t/s)
    else:
        ai = (coupon/2)*((s-t)/s)

    v = 1/(1+(ytm/2))
    cf = np.zeros(n, dtype=float)
    e = np.zeros(n, dtype=float)

    for k in range(n):
        if k==0:
            if is_ex:
                cf[k] = 0
            else:
                cf[k] = (coupon/2)
            if n == 1:
                cf[k] += 100
        elif k == n-1:
            cf[k] = (coupon/2)+100
        else:
            cf[k] = (coupon/2)
        e[k] = ((t/s)+k)
    
    dirty_P = np.sum(cf*v**e)
    clean_P = dirty_P-ai

    dv01 = 1e-4*0.5*np.sum(e*cf*v**(e+1))
    mac_dur = np.sum((e/2)*cf*v**e)/dirty_P
    mod_dur = mac_dur/(1+(ytm/2))

    return {"clean": clean_P, "dirty": dirty_P,\
            "ai": ai, "dv01": dv01,\
            "mac_dur": mac_dur, "mod_dur": mod_dur}

In [202]:
def bump_reprice(settle, maturity, coupon, ytm, ex_div_days):
    """Function to test whether (P(y-1bp)-P(y+1bp))/2==DV01"""
    p_up = gilt_price(coupon=coupon, ytm=ytm+1e-4, ex_div_bdays=ex_div_days,\
            settle=settle, maturity=maturity)["dirty"]
    p_down = gilt_price(coupon=coupon, ytm=ytm-1e-4, ex_div_bdays=ex_div_days,\
                settle=settle, maturity=maturity)["dirty"]
    
    return (p_down-p_up)/2

# MODULE 1

In [203]:
# TEST
coupon, ytm, ex_div_bdays, settle, maturity = 4.875, 0.05383, 7, date(2026, 11, 26), date(2055, 12, 7) 
results = gilt_price(coupon=coupon, ytm=ytm, ex_div_bdays=ex_div_bdays,\
                settle=settle, maturity=maturity)
results

{'clean': np.float64(92.58397657966573),
 'dirty': np.float64(92.4374601862231),
 'ai': -0.14651639344262293,
 'dv01': np.float64(0.13794847549824657),
 'mac_dur': np.float64(15.325102878302047),
 'mod_dur': np.float64(14.923438530260096)}

In [204]:
coupon, ytm, ex_div_bdays, settle, maturity = 4.25, 0.05, 7, date(2026, 9, 30), date(2036, 7, 31) 
results = gilt_price(coupon=coupon, ytm=ytm, ex_div_bdays=ex_div_bdays,\
                settle=settle, maturity=maturity)

# MODULE 2

In [205]:
spot_short_end = pd.read_excel(os.path.join("../data/interim_data", "glc_nominal/2016_to_2024.xlsx"),
                            sheet_name="spot_short_end", parse_dates=["years:"])
spot_curve = pd.read_excel(os.path.join("../data/interim_data", "glc_nominal/2016_to_2024.xlsx"),
                            sheet_name="spot_curve", parse_dates=["years:"])

In [216]:
dmo_ref_price = pd.read_excel(os.path.join("../data/interim_data", "dmo_ref_price/2016.xls"),
                            sheet_name="Sheet1", parse_dates=["Close of Business Date", "Redemption Date"])

In [207]:
short_df = spot_short_end[spot_short_end["years:"]=="2016-03-15"]
long_df = spot_curve[spot_curve["years:"]=="2016-03-15"]

In [208]:
def cash_flows(settle, maturity, coupon, ex_div_bdays=7):
    """Remaining cash flows after `settle`, on NOMINAL (unadjusted) coupon dates.
    Returns (dates, amounts, prev_cpn, is_ex). Single source of truth for schedule + ex-div."""
    if settle >= maturity:
        raise ValueError("settle must be before maturity")
    dates, k = [], 0
    while maturity - relativedelta(months=6 * k) > settle:      # step back from maturity, no drift
        dates.append(maturity - relativedelta(months=6 * k))
        k += 1
    dates = dates[::-1]
    prev_cpn = maturity - relativedelta(months=6 * k)
    ex_div_date = (pd.Timestamp(dates[0]) - pd.offsets.BDay(ex_div_bdays)).date()
    is_ex = settle >= ex_div_date          # TODO: Exercise A will tell you whether this should be > or >=
    amounts = np.full(len(dates), coupon / 2)
    amounts[-1] += 100
    if is_ex:
        amounts[0] -= coupon / 2
        
    return dates, amounts, prev_cpn, is_ex

In [209]:
def curve_on(date, short_df, long_df):
    """z(t) in % for one date. short_df/long_df: full sheets, dates as index, maturities as columns."""
    def row(df):
        s = df.loc[pd.Timestamp(date)].dropna()
        s.index = pd.Series(s.index.astype(str)).str.extract(r"([\d\.]+)")[0].astype(float).values
        return s
    def parse_maturities(series):
        clean_index = pd.Series(series.index.astype(str)).str.extract(r"([\d\.]+)")[0].astype(float)
        series = series.copy()
        series.index = clean_index
        return series

    s = short_df.iloc[0]
    l = long_df.iloc[0]
    s, l = parse_maturities(s), parse_maturities(l)

    pts = pd.concat([s[s.index <= 5.0].dropna(), l[l.index > 5.0].dropna()]).sort_index()
    pts = pts[~pts.index.duplicated(keep="first")]
    t_knots, z_knots = pts.index.to_numpy(float), pts.to_numpy(float)
    spline = CubicSpline(t_knots, z_knots, bc_type="natural")
    t_min, t_max = t_knots[0], t_knots[-1]                     # flat beyond the actual data ends
    
    return lambda t: spline(np.clip(np.asarray(t, float), t_min, t_max))

In [210]:
def model_dirty(settle, maturity, coupon, z):
    """PV of the gilt's remaining cash flows on the curve z(t)"""
    dates, amounts, _, _ = cash_flows(settle, maturity, coupon)
    t = np.array([(d - settle).days / 365.25 for d in dates])
    
    return float(np.sum(amounts * np.exp(-z(t) / 100 * t)))

In [211]:
def model_yield(settle, maturity, coupon, z):
    target = model_dirty(settle, maturity, coupon, z)          # <- uses the gilt's own coupon
    
    return brentq(lambda y: gilt_price(settle=settle, maturity=maturity, coupon=coupon,
                                       ytm=y, ex_div_bdays=7)["dirty"] - target, -0.05, 0.30)

In [212]:
coupon, ytm, ex_div_bdays, settle, maturity = 4.5, 0.05, 7, date(2016, 3, 16), date(2034, 9, 7)
z = curve_on(date=settle, short_df=short_df, long_df=long_df)
y_model = model_yield(settle=settle, maturity=maturity, coupon=coupon, z=z)
y_dmo = 2.1334
residual = round((y_dmo - y_model*100)*100, 2)
print(f"model yield: {round(y_model*100, 4)}\tresidual: {residual}")

model yield: 2.1195	residual: 1.39


In [213]:
coupon, ytm, ex_div_bdays, settle, maturity = 1.5, 0.05, 7, date(2016, 3, 16), date(2026, 7, 22)
z = curve_on(date=settle, short_df=short_df, long_df=long_df)
y_model = model_yield(settle=settle, maturity=maturity, coupon=coupon, z=z)
y_dmo = 1.6477
residual = round((y_dmo - y_model*100)*100, 2)
print(f"model yield: {round(y_model*100, 4)}\tresidual: {residual}")

model yield: 1.6206	residual: 2.71


In [214]:
coupon, ytm, ex_div_bdays, settle, maturity = 4.25, 0.05, 7, date(2016, 3, 16), date(2055, 12, 7)
z = curve_on(date=settle, short_df=short_df, long_df=long_df)
y_model = model_yield(settle=settle, maturity=maturity, coupon=coupon, z=z)
y_dmo = 2.1626
residual = round((y_dmo - y_model*100)*100, 2)
print(f"model yield: {round(y_model*100, 4)}\tresidual: {residual}")

model yield: 2.1817	residual: -1.91


## Exercise A

In [221]:
dmo_ref_price.rename(columns={"Gilt Name": "gilt",
                              "ISIN Code": "ISIN",
                              "Redemption Date": "maturity",
                              "Close of Business Date": "date",
                              "Clean Price\n(£)": "clean_price",
                              "Dirty Price\n(£)": "dirty_price",
                              "Accrued Interest (£)": "ai",
                              "Yield \n(%)": "yield",
                              "Modified \nDuration": "mod_duration"},
                    inplace=True)
dmo_ref_price.head()

,gilt,ISIN,maturity,date,clean_price,dirty_price,ai,yield,mod_duration
0,0 1/8% Index-linked Treasury Gilt 2019,GB00BBDR7T29,2019-11-22,2016-01-04,105.07,109.179226,0.015699,-1.149312,3.89
1,0 1/8% Index-linked Treasury Gilt 2019,GB00BBDR7T29,2019-11-22,2016-01-05,105.18,109.298076,0.016056,-1.177101,3.89
2,0 1/8% Index-linked Treasury Gilt 2019,GB00BBDR7T29,2019-11-22,2016-01-06,105.33,109.458497,0.016413,-1.214664,3.89
3,0 1/8% Index-linked Treasury Gilt 2019,GB00BBDR7T29,2019-11-22,2016-01-07,105.31,109.442286,0.016771,-1.210723,3.89
4,0 1/8% Index-linked Treasury Gilt 2019,GB00BBDR7T29,2019-11-22,2016-01-08,105.45,109.600429,0.017843,-1.247808,3.88


In [222]:
print(f"Before droping NA: {len(dmo_ref_price)}")
dmo_ref_price.dropna(subset=["ISIN", "date"], inplace=True)
print(f"After droping NA: {len(dmo_ref_price)}")

Before droping NA: 56364
After droping NA: 56364


In [223]:
is_linker = dmo_ref_price["gilt"].str.contains("Index-linked", case=False)
is_strip  = dmo_ref_price["gilt"].str.contains("Strip", case=False)
conv = dmo_ref_price[~is_linker & ~is_strip]                  # 28 linkers + 155 strips removed → 41 conventionals
print(f"Count of conventional GILTS: {len(conv)}")

Count of conventional GILTS: 10510


In [227]:
conv.sort_values("maturity").head(50)

,gilt,ISIN,maturity,date,clean_price,dirty_price,ai,yield,mod_duration
8501,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-07,100.06,100.983913,0.923913,0.419142,0.04
8498,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-04,100.08,100.987609,0.907609,0.265772,0.05
8499,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-05,100.07,100.983043,0.913043,0.386544,0.04
8500,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-06,100.07,100.988478,0.918478,0.280080,0.04
8502,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-08,100.05,100.990217,0.940217,0.324311,0.03
8503,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-11,100.04,100.985652,0.945652,0.523493,0.03
8505,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-13,100.00,100.000000,0.000000,0.000000,0.00
8504,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-12,100.04,100.991087,0.951087,0.361178,0.02
8507,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-15,100.00,100.000000,0.000000,0.000000,0.00
8508,2% Treasury Gilt 2016,GB00B3QCG246,2016-01-22,2016-01-18,100.00,100.000000,0.000000,0.000000,0.00
